In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score
import pickle

In [2]:
input_file = "/Users/ahb232/Desktop/machine_learning/zero_shot_assessment/model_8192_l24/scores/Zmays_833_Zm-B73_scores.tsv"
training_data = pd.read_csv(input_file, sep="\t")

/var/folders/p7/0ybzj5252tjb8x2gtktw7qghrw3sgy/T/ipykernel_57939/2898029223.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  training_data = pd.read_csv(input_file, sep="\t")


In [10]:
def to_average(x):
    y = [float(y) for y in x.split(",")]
    return sum(y) / len(y)

In [4]:
training_single_exon = training_data[training_data["donor"].isna()]
training_multi_exon = training_data[training_data["donor"].notna()]

In [7]:
multi_exon_x = np.array(
    [
        list(training_multi_exon["TIS"]),
        list(training_multi_exon["TTS"]),
        [to_average(x) for x in training_multi_exon["donor"]],
        [to_average(x) for x in training_multi_exon["acceptor"]],
    ]
).transpose()
multi_exon_y = np.array([1 if x else 0 for x in training_multi_exon["classical"]])

single_exon_x = np.array(
    [list(training_single_exon["TIS"]), list(training_single_exon["TTS"])]
).transpose()
single_exon_y = np.array([1 if x else 0 for x in training_single_exon["classical"]])

In [8]:
def fit_model(train_x, train_y, positive_rate=0.75):
    # Adapted from https://github.com/hkiyomaru/pu-learning
    _clf = LogisticRegression().fit(train_x, train_y)

    train_x_labeled = train_x[train_y == 1]
    train_x_unlabeled = train_x[train_y == 0]

    c = train_x_labeled.shape[0] / (
        train_x_labeled.shape[0] + train_x_unlabeled.shape[0] * positive_rate
    )

    train_ss_prob_unlabeled = _clf.predict_proba(train_x_unlabeled)[:, 1]

    new_train_x = []
    new_train_y = []
    sample_weight = []

    # Labeled data is used as positive (y=1)
    for x_labeled in train_x_labeled:
        new_train_x.append(x_labeled)
        new_train_y.append(1)
        sample_weight.append(1)

    # Unlabeled data is used as positive (y=1)
    for x_unlabeled, train_s_prob_unlabeled in zip(
        train_x_unlabeled, train_ss_prob_unlabeled
    ):
        new_train_x.append(x_unlabeled)
        new_train_y.append(1)
        w_pos = ((1 - c) / c) * (train_s_prob_unlabeled / (1 - train_s_prob_unlabeled))
        sample_weight.append(w_pos)

    # Unlabeled data is used as negative as well (y=0)
    for x_unlabeled, train_s_prob_unlabeled in zip(
        train_x_unlabeled, train_ss_prob_unlabeled
    ):
        new_train_x.append(x_unlabeled)
        new_train_y.append(0)
        w_pos = ((1 - c) / c) * (train_s_prob_unlabeled / (1 - train_s_prob_unlabeled))
        w_neg = 1 - w_pos
        sample_weight.append(w_neg)

    clf = LogisticRegression().fit(
        new_train_x, new_train_y, sample_weight=sample_weight
    )

    return clf

# train for multi-exon transcripts

In [10]:
test_indices = np.random.choice(np.arange(multi_exon_y.shape[0]), 6260, replace=False)
test_x = multi_exon_x[test_indices,]
train_x = np.delete(multi_exon_x, test_indices, axis=0)
test_y = multi_exon_y[test_indices]
train_y = np.delete(multi_exon_y, test_indices)

In [11]:
multi_exon_model = fit_model(train_x, train_y)

In [12]:
test_y_hat = multi_exon_model.predict(test_x)
test_y_prob = multi_exon_model.predict_proba(test_x)[:, 1]

In [13]:
recall_score(test_y, test_y_hat)

0.9393939393939394

In [14]:
sum(np.logical_and(test_y_hat, test_y))

np.int64(93)

In [15]:
sum(test_y_hat) / len(test_y_hat)

np.float64(0.7523961661341853)

# train for single-exon transcripts

In [17]:
test_indices = np.random.choice(np.arange(single_exon_y.shape[0]), 994, replace=False)
test_x = single_exon_x[test_indices,]
train_x = np.delete(single_exon_x, test_indices, axis=0)
test_y = single_exon_y[test_indices]
train_y = np.delete(single_exon_y, test_indices)

In [18]:
single_exon_model = fit_model(train_x, train_y)

In [20]:
test_y_hat = single_exon_model.predict(test_x)
test_y_prob = single_exon_model.predict_proba(test_x)[:, 1]

In [21]:
recall_score(test_y, test_y_hat)

0.75

In [22]:
sum(np.logical_and(test_y_hat, test_y))

np.int64(3)

In [23]:
sum(test_y_hat) / len(test_y_hat)

np.float64(0.761569416498994)

# Test models against a cross-species dataset

In [7]:
input_file = "/Users/ahb232/Desktop/machine_learning/zero_shot_assessment/model_8192_l24/scores/Slycopersicum_all_scores_pc2.tsv"
validation_data = pd.read_csv(input_file, sep="\t")

In [32]:
valid_single_exon_df = validation_data[validation_data["donor"].isna()]
valid_multi_exon_df = validation_data[validation_data["donor"].notna()]

In [11]:
valid_multi_exon_x = np.array(
    [
        list(valid_multi_exon_df["TIS"]),
        list(valid_multi_exon_df["TTS"]),
        [to_average(x) for x in valid_multi_exon_df["donor"]],
        [to_average(x) for x in valid_multi_exon_df["acceptor"]],
    ]
).transpose()
valid_multi_exon_y = np.array([1 if x else 0 for x in valid_multi_exon_df["classical"]])

valid_single_exon_x = np.array(
    [list(valid_single_exon_df["TIS"]), list(valid_single_exon_df["TTS"])]
).transpose()
valid_single_exon_y = np.array(
    [1 if x else 0 for x in valid_single_exon_df["classical"]]
)

In [12]:
valid_multi_exon = multi_exon_model.predict(valid_multi_exon_x)
valid_single_exon = single_exon_model.predict(valid_single_exon_x)

In [13]:
recall_score(valid_multi_exon_y, valid_multi_exon)

0.9900990099009901

In [14]:
recall_score(valid_single_exon_y, valid_single_exon)

0.9

In [31]:
sum(valid_multi_exon) / len(valid_multi_exon)

np.float64(0.7196711909768687)

In [32]:
sum(valid_single_exon) / len(valid_single_exon)

np.float64(0.7859848484848485)

In [33]:
sum(valid_single_exon_y)

np.int64(20)

In [30]:
sum(np.logical_and(np.logical_not(valid_single_exon), valid_single_exon_y))

np.int64(2)

In [5]:
with open("multi_exon_model.pkl", "rb") as file:
    multi_exon_model = pickle.load(file)
with open("single_exon_model.pkl", "rb") as file:
    single_exon_model = pickle.load(file)

In [17]:
valid_proba_multi = multi_exon_model.predict_proba(valid_multi_exon_x)[:, 1]
valid_proba_single = single_exon_model.predict_proba(valid_single_exon_x)[:, 1]

In [24]:
out_df_multi = pd.DataFrame(
    {"label": valid_multi_exon_y, "predict": valid_proba_multi, "type": "multi"}
)
out_df_single = pd.DataFrame(
    {"label": valid_single_exon_y, "predict": valid_proba_single, "type": "single"}
)

out_df = pd.concat((out_df_multi, out_df_single))

In [27]:
out_df.to_csv("tomato_predictions.tsv", sep="\t", index=False)

In [38]:
valid_multi_exon_df

,chrom,gene,transcript,start,end,strand,donor,acceptor,longest,TIS,TTS,classical,predict
0,SL4.0ch00,Solyc00g005280,Solyc00g005280.1.1,1075279,1075781,-,0.44406396,0.36569816,True,0.732243,0.650884,False,1
1,SL4.0ch00,Solyc00g006650,Solyc00g006650.1.1,8739396,8751154,+,"0.827764,0.91286755","0.89063275,0.92157865",True,0.841816,0.465268,False,1
2,SL4.0ch00,Solyc00g007330,Solyc00g007330.1.1,2379603,2380806,-,0.86062115,0.67213875,True,0.286229,0.249753,False,0
4,SL4.0ch00,Solyc00g007350,Solyc00g007350.1.1,2416398,2417839,-,"0.88358,0.9667393,0.13700512","0.4141978,0.28883195,0.84285295",True,0.261645,0.271894,False,0
5,SL4.0ch00,Solyc00g007450,Solyc00g007450.1.1,5306616,5307485,-,"0.19448018,0.6362045","0.2629807,0.8399335",True,0.457376,0.915170,False,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
34070,SL4.0ch12,Solyc12g150128,Solyc12g150128.1.1,63979663,63988157,-,"0.9852161,0.98176324","0.9884887,0.9695042",True,0.660945,0.266822,False,1
34071,SL4.0ch12,Solyc12g150129,Solyc12g150129.1.1,63999950,64000448,-,0.9887593,0.57625055,True,0.975129,0.717162,False,1
34072,SL4.0ch12,Solyc12g150130,Solyc12g150130.1.1,64001453,64001925,-,0.98666394,0.96253514,True,0.979609,0.647568,False,1
34073,SL4.0ch12,Solyc12g150131,Solyc12g150131.1.1,65690362,65693362,-,"0.94064635,0.99873245","0.62123394,0.8567238",True,0.774861,0.927678,False,1


In [48]:
validation_data

,chrom,gene,transcript,start,end,strand,donor,acceptor,longest,TIS,TTS,classical,predict
0,SL4.0ch00,Solyc00g007340,Solyc00g007340.1.1,2388673,2389041,-,NaN,NaN,True,0.401326,0.244927,False,1
1,SL4.0ch00,Solyc00g011670,Solyc00g011670.1.1,1818915,1819097,+,NaN,NaN,True,0.980932,0.658258,False,1
2,SL4.0ch00,Solyc00g013100,Solyc00g013100.1.1,3045076,3045312,+,NaN,NaN,True,0.236244,0.231657,False,0
3,SL4.0ch00,Solyc00g013110,Solyc00g013110.1.1,3045979,3046245,-,NaN,NaN,True,0.317963,0.338112,False,1
4,SL4.0ch00,Solyc00g013170,Solyc00g013170.1.1,3069442,3069651,+,NaN,NaN,True,0.782372,0.354621,False,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
34070,SL4.0ch12,Solyc12g150128,Solyc12g150128.1.1,63979663,63988157,-,"0.9852161,0.98176324","0.9884887,0.9695042",True,0.660945,0.266822,False,1
34071,SL4.0ch12,Solyc12g150129,Solyc12g150129.1.1,63999950,64000448,-,0.9887593,0.57625055,True,0.975129,0.717162,False,1
34072,SL4.0ch12,Solyc12g150130,Solyc12g150130.1.1,64001453,64001925,-,0.98666394,0.96253514,True,0.979609,0.647568,False,1
34073,SL4.0ch12,Solyc12g150131,Solyc12g150131.1.1,65690362,65693362,-,"0.94064635,0.99873245","0.62123394,0.8567238",True,0.774861,0.927678,False,1


In [47]:
validation_data = pd.concat(
    (valid_single_exon_df, valid_multi_exon_df), ignore_index=True
)

In [49]:
validation_data[
    np.logical_and(validation_data["classical"], validation_data["predict"] == 0)
]

,chrom,gene,transcript,start,end,strand,donor,acceptor,longest,TIS,TTS,classical,predict
5342,SL4.0ch08,gene:Solyc08g076930.1,Solyc08g076930.1.1,58981111,58983180,+,NaN,NaN,True,0.274633,0.308082,True,0
6199,SL4.0ch10,gene:Solyc10g007960.1,Solyc10g007960.1.1,1999750,2001225,-,NaN,NaN,True,0.355726,0.255655,True,0
10070,SL4.0ch01,Solyc01g090430,Solyc01g090430.3.1,76358431,76363350,+,"0.8429978,0.98022467","0.58228725,0.2964691",True,0.320014,0.349720,True,0


In [50]:
validation_data[
    np.logical_and(validation_data["classical"], validation_data["predict"] == 1)
]

,chrom,gene,transcript,start,end,strand,donor,acceptor,longest,TIS,TTS,classical,predict
550,SL4.0ch01,gene:Solyc01g087850.2,Solyc01g087850.2.1,74964686,74967793,+,NaN,NaN,True,0.997570,0.396504,True,1
1081,SL4.0ch01,Solyc01g111170,Solyc01g111170.3.1,89876815,89877825,+,NaN,NaN,True,0.999325,0.487521,True,1
1654,SL4.0ch02,gene:Solyc02g092430.1,Solyc02g092430.1.1,51526866,51527306,+,NaN,NaN,True,0.607247,0.396155,True,1
1692,SL4.0ch02,Solyc02g021400,Solyc02g021400.2.1,21003695,21005687,+,NaN,NaN,True,0.962933,0.721182,True,1
2000,SL4.0ch03,gene:Solyc03g044890.1,Solyc03g044890.1.1,10056458,10057255,-,NaN,NaN,True,0.871187,0.524015,True,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
33647,SL4.0ch12,Solyc12g070100,Solyc12g070100.2.1,29206256,29228021,-,"0.9990678,0.9988849,0.9991905,0.9787403,0.9744...","0.9996762,0.89979076,0.9995222,0.30662593,0.99...",True,0.383202,0.681759,True,1
33831,SL4.0ch12,Solyc12g096150,Solyc12g096150.2.1,64703131,64705801,+,"0.34479776,0.9961648,0.99362224,0.9765176","0.2126389,0.98216254,0.99950576,0.99962974",True,0.994745,0.794308,True,1
33858,SL4.0ch12,Solyc12g096540,Solyc12g096540.2.1,64933408,64936053,-,"0.99257135,0.99748254,0.9910767,0.99825525","0.99929345,0.99967504,0.99940765,0.99933404",True,0.985973,0.561323,True,1
33871,SL4.0ch12,Solyc12g096700,Solyc12g096700.2.1,65031805,65034155,-,"0.99891275,0.9996621","0.99970555,0.9995258",True,0.991010,0.817941,True,1


In [40]:
multi_exon_model.coef_

array([[1257.14461103,  887.72821422,  571.2411655 ,  846.96884891]])

In [42]:
multi_exon_model.intercept_

array([-1902.2387911])

In [43]:
single_exon_model.coef_

array([[11.81083443, 10.79526443]])

In [44]:
single_exon_model.intercept_

array([-7.26727317])